# 🏗️ 회귀 세션 과제

실습에서는 모델을 만들고, `summary()`를 읽고, 가정을 검정하고, 잔차를 들여다봤었죠?

이번 과제에서는 이렇게 따져본 결과를 가지고, 모델을 실제로 고쳐볼 예정입니다.



---

## 진행 방식

- 막히면 **▶ 접힌 힌트**를 펼쳐 보셔도 괜찮습니다.
- 모델을 비교할 때는 **항상 테스트 데이터 기준**으로 봐야 합니다!

In [ ]:
# 들어가기 전, 그래프의 한글 깨짐을 방지하는 코드입니다. 실행해 주세요!

import platform
import matplotlib.pyplot as plt

# OS별 한글 폰트 설정
if platform.system() == "Darwin":  # Mac OS
    plt.rc("font", family="AppleGothic")
elif platform.system() == "Windows":  # Windows
    plt.rc("font", family="Malgun Gothic")

# 마이너스(-) 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

---

# Part A

## A-1. 데이터 살펴 보기

### 콘크리트 압축강도 데이터

콘크리트는 건설·토목 분야에서 가장 중요한 재료 중 하나예요.

그중 **압축강도(compressive strength)** 는 재료 성능을 평가하는 핵심 지표인데,
이 값은 **어떤 재료를 얼마나 섞었는지** 와 **얼마나 오래 굳혔는지** 에 따라 달라집니다.

우리의 목표는 **배합 정보로 압축강도를 예측하는 모델**을 만드는 겁니다!

**독립변수** (단위: kg/m³)

| 칼럼 | 뜻 | | 칼럼 | 뜻 |
| --- | --- | --- | --- | --- |
| `cement` | 시멘트 | | `superplasticizer` | 고성능 감수제 |
| `slag` | 고로슬래그 | | `coarseaggregate` | 굵은 골재 |
| `flyash` | 플라이애시 | | `fineaggregate` | 잔골재 |
| `water` | 물 | | | |

| 칼럼 | 뜻 | 단위 |
| --- | --- | --- |
| `age` | **양생 기간** (굳힌 기간) | 일 (1 ~ 365) |

**종속변수**

- `csMPa` : 압축강도 (MPa)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = pd.read_csv('./Concrete_data.csv')   # 경로가 다르면 맞게 수정해 주세요
data.head()

In [ ]:
# 데이터의 크기와 칼럼별 자료형·결측치를 확인해 봅시다.
print('데이터 크기:', **)
data.**()

### 중복 데이터 확인하기

결측치는 없지만, **똑같은 배합이 두 번 기록된 행**이 있을 수 있어요.

중복이 남아 있으면 그 배합에만 모델이 과하게 맞춰질 수 있으니 확인하고 정리해 줍시다.

In [ ]:
# 중복 행이 몇 개인지 확인하고, 제거해 봅시다.
print('중복 행 개수:', data.**().sum())

data = data.**().reset_index(drop=True)
print('정리 후 크기:', data.shape)

## A-2. Baseline 모델 만들기

먼저 **아무 손질도 하지 않은 상태**로 선형회귀를 돌려 봅니다.
이걸 **베이스라인(baseline)** 모델이라고 불러요.
앞으로 만들 모델들이 이보다 나은지 재는 **기준선**이 됩니다.

> 💡 **스케일링은 아직 하지 않고 넘어가겠습니다.** 일반 선형회귀는 단위가 달라도 결과가 같거든요.
> 스케일링이 꼭 필요해지는 건 규제 모델부터예요!

In [ ]:
X = data.drop(columns=**)
y = data[**]

# 전체의 20%를 테스트용으로 떼어 둡시다. random_state는 42로 고정해 주세요.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=**, random_state=42
)

print(f'train: {len(X_train)}개 / test: {len(X_test)}개')

In [ ]:
# 선형회귀 모델을 만들고 학습시킨 뒤, 테스트 데이터로 예측해 봅시다.
base = LinearRegression().**(X_train, y_train)
pred_base = base.**(X_test)

def report(name, model_pred, y_true=None):
    y_true = y_test if y_true is None else y_true
    return {
        '모델': name,
        'test R2': round(r2_score(y_true, model_pred), 4),
        'test MAE': round(mean_absolute_error(y_true, model_pred), 3),
        'test RMSE': round(np.sqrt(mean_squared_error(y_true, model_pred)), 3),
    }

scores = [report('A. baseline (손질 없음)', pred_base)]

print(f'train R2: {base.score(X_train, y_train):.4f}')
display(pd.DataFrame(scores))

**Q.** MAE 값을 해석해 봅시다. `csMPa`의 단위가 MPa라는 점을 생각하면, 이 모델은 평균적으로 얼마나 빗나가고 있나요?
그리고 R²만 보고 판단하지 않고 **MAE를 함께 보는 이유**는 뭘까요?

**A.**

## A-3. 문제 찾기

R²가 0.58 정도 나왔을 겁니다. 나쁘지는 않지만 썩 만족스럽지도 않은 값이네요.
**어디가 문제인지** 잔차를 통해 찾아봅시다!

In [ ]:
# 잔차를 구하고, 예측값(x축) 대비 잔차(y축) 산점도를 그려 봅시다.
resid = ** - **

plt.figure(figsize=(8, 5))
plt.scatter(**, **, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('예측값'); plt.ylabel('잔차')
plt.title('Residuals vs Fitted')
plt.show()

패턴이 보이시나요? 그럼 **어느 변수가 원인인지** 찾아봅시다.
각 독립변수와 `csMPa`의 관계를 한 번에 그려볼게요.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), X.columns):
    ax.scatter(data[col], y, alpha=0.3, s=12)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 🤔 잠깐, `age` 그래프가 좀 이상하지 않나요?

여덟 칸 중 `age`만 유독 **세로 줄무늬**처럼 보입니다. 이유가 있어요.

- `age`는 값이 **14가지뿐**입니다. (1, 3, 7, 14, 28, 56, 90, 91, 100, 120, 180, 270, 360, 365일)
  연속적으로 잰 게 아니라 **정해진 시점에만 측정**했거든요. 그래서 같은 x 위에 점이 세로로 쌓입니다.
- 게다가 **데이터의 73%가 28일 이내**에 몰려 있는데 x축은 365까지 뻗어 있어서, 정작 봐야 할 왼쪽이 뭉개져 보여요.

이대로는 관계를 읽기 어렵겠죠? `age`만 따로 떼어 **양생기간별 평균을 겹쳐서** 다시 그려보겠습니다.

In [ ]:
# 그냥 실행해 주세요!
# age만 따로 떼어, 양생기간별 평균(빨간 점)을 겹쳐 그립니다.

age_mean = data.groupby('age')['csMPa'].agg(['mean', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, use_log, title, xlabel in [
    (axes[0], False, '(1) Original scale', 'age (days)'),
    (axes[1], True,  '(2) Log scale',      'age (days, log scale)'),
]:
    ax.scatter(data['age'], data['csMPa'], alpha=0.18, s=14, color='gray', label='raw data')
    ax.plot(age_mean.index, age_mean['mean'], '-', color='crimson', lw=1.5, zorder=3)
    ax.scatter(age_mean.index, age_mean['mean'], s=age_mean['count'] * 1.5,
               color='crimson', zorder=4, label='mean by age\n(size = n)')
    if use_log:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel); ax.set_ylabel('csMPa'); ax.set_title(title)
    ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

display(age_mean.round(1).T)

> 💡 **빨간 점의 크기 = 그 기간에 측정된 데이터 개수**입니다.
> 점이 크면 믿을 만한 평균이고, 작으면 몇 개 안 되는 값이라 들쭉날쭉할 수 있어요.
>
> 91일 지점이 유독 튀어 올라 있죠? **데이터가 17개뿐이라** 그렇습니다.
> 표를 보시면 56일 이후로 표본이 확 줄어드는 걸 확인할 수 있어요.

**Q.** 위 두 그래프를 보고 답해 주세요.

1. **왼쪽 그래프**에서 빨간 선을 따라가 보세요.
   **1~28일 구간**과 **100~365일 구간** 중, 평균이 더 가파르게 오르는 쪽은 어디인가요?
2. **오른쪽 그래프**(x축만 로그로 바꾼 것)에서 빨간 선은 어떤 모양에 가까워졌나요?
3. 그렇다면 `age`와 `csMPa`의 관계를 **직선 하나로** 표현해도 괜찮을까요?

**A.**

<details>
<summary><b>▶ 힌트</b></summary>
<br>

- **1번**: 큰 빨간 점 몇 개만 읽어보세요. 3일에 18.4, 28일에 36.4, 365일에 43.6입니다.
  **25일 동안 오른 폭**과 **337일 동안 오른 폭** 중 어느 쪽이 클까요?
- **2번**: x축만 바꿨을 뿐인데 구불구불하던 선이 어떻게 변했는지 보세요.
- **3번**: 로그 축에서 직선이 된다는 건, 원래 축에서는 어떤 모양이라는 뜻일까요?

</details>

---

# Part B

다음으로는 아래와 같은 네 가지 방법으로 모델을 개선해 봅니다!

| | 시도 | 물어볼 것 |
| --- | --- | --- |
| B-1 | `age` 변수 변환 | 왜 하필 로그일까? |
| B-2 | 파생변수 만들기 | 도메인 지식인데 왜 효과가 작을까? |
| B-3 | 규제선형모델 | 같은 규제인데 왜 릿지와 라쏘가 다를까? |
| B-4 | 차원 축소(PCA) | 분산의 95%를 설명하는데 왜 성능이 나빠질까? |


## B-1. `age`를 로그로 바꿔보기

Part A에서 찾은 문제를 고쳐봅시다.

$$ y = a + b \cdot age \quad \rightarrow \quad y = a + b \cdot \log(age) $$

> 💡 **주의** 변환은 항상 train data와 test data, 즉 `X_train`과 `X_test`에 **똑같이** 적용해야 합니다.
> 학습할 때 쓴 형태와 예측할 때 쓴 형태가 다르면 모델이 엉뚱한 값을 내놓을 수 있기 때문이에요.

In [ ]:
X_train_log = X_train.copy()
X_test_log = X_test.copy()

# age 칼럼에 자연로그를 씌워 봅시다. (train과 test 모두!)
X_train_log['age'] = np.**(X_train_log['age'])
X_test_log['age'] = np.**(X_test_log['age'])

m_log = LinearRegression().fit(X_train_log, y_train)
pred_log = m_log.predict(X_test_log)

scores.append(report('B-1. age 로그 변환', pred_log))
print(f'train R2: {m_log.score(X_train_log, y_train):.4f}')
display(pd.DataFrame(scores))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(data['age'], y, alpha=0.3, s=14)
axes[0].set_xlabel('age'); axes[0].set_ylabel('csMPa'); axes[0].set_title('변환 전')
axes[1].scatter(np.log(data['age']), y, alpha=0.3, s=14, color='tab:orange')
axes[1].set_xlabel('log(age)'); axes[1].set_ylabel('csMPa'); axes[1].set_title('변환 후')
plt.tight_layout(); plt.show()

**Q.** R²가 얼마나 올랐나요? 그리고 위 두 그래프를 보면서, **로그 변환의 효과가 무엇일지** 설명해 봅시다!

**A.**

<details>
<summary><b>▶ 그래프를 어떻게 볼까요</b></summary>
<br>

왼쪽 그래프에서 점들이 **왼쪽 끝에 잔뜩 뭉쳐 있고** 오른쪽은 텅 비어 있죠?

오른쪽 그래프에서는 점들이 어떻게 배치되어 있나요?

로그 함수는 **작은 값들 사이는 넓게, 큰 값들 사이는 좁게** 펴주는 성질이 있습니다.

</details>

## B-2. 파생변수 만들기

콘크리트 공학에는 잘 알려진 지표가 있어요. 바로 **물-시멘트비(w/c ratio)** 입니다.

$$ w/c\ ratio = \frac{water}{cement} $$

물이 많을수록 반죽은 다루기 쉬워지지만 굳고 나면 약해지고, 시멘트가 많을수록 단단해집니다.
그래서 이 **비율**이 강도를 좌우한다고 알려져 있어요.

파생변수를 만들 때에는 보통 도메인 지식을 많이 활용합니다.
우리도 이 도메인 지식을 가지고 파생변수를 만들어 봐요!

In [ ]:
X_train_wc = X_train_log.copy()
X_test_wc = X_test_log.copy()

# water를 cement로 나눈 wc_ratio 칼럼을 만들어 봅시다. (train과 test 모두!)
X_train_wc['wc_ratio'] = **
X_test_wc['wc_ratio'] = **

m_wc = LinearRegression().fit(X_train_wc, y_train)
pred_wc = m_wc.predict(X_test_wc)

scores.append(report('B-2. + w/c ratio', pred_wc))
display(pd.DataFrame(scores))

지표가 거의 오르지 않았습니다. 왜일까요? **VIF를 한번 확인해 봅시다.**

In [ ]:
def vif_of(df_X):
    Xc = sm.**(df_X)          # 실습에서 배운 그 함수! 잊지 않으셨죠?
    s = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])],
                  index=Xc.columns)
    return s.drop('const').sort_values(ascending=False)

compare_vif = pd.DataFrame({
    'w/c 추가 전': vif_of(X_train_log).round(2),
    'w/c 추가 후': vif_of(X_train_wc).round(2),
})
display(compare_vif)

**Q.** 아래의 질문에 답해 주세요.

1. `wc_ratio`를 추가했더니 **`cement`의 VIF**가 어떻게 변했나요?
2. 도메인 지식으로 만든 좋은 변수인데 왜 성능이 거의 오르지 않았을까요?

**A.**

<details>
<summary><b>▶ 참고</b></summary>
<br>

`wc_ratio`는 `water`와 `cement`로 만들어졌습니다.

그런데 `water`와 `cement`는 **이미 모델 안에 들어 있어요.**

모델 입장에서 `wc_ratio`는 **새로운 정보일지, 이미 아는 정보를 다르게 표현한 것일지 생각해 봅시다.**

</details>

> 🤔 **그래서 `wc_ratio`는 써야 하나요, 말아야 하나요?**
>
> 성능이 0.003 오르는 대신 다중공선성이 심해졌으니, **이번엔 쓰지 않는 편**이 낫겠습니다.

> 앞으로는 **B-1의 모델(`age` 로그 변환)** 을 기준으로 계속 진행할게요.

## B-3. 규제선형모델 — 릿지와 라쏘

실습에서 릿지 모델을 살짝 맛봤었습니다.
`alpha`를 키우면 회귀계수가 눌려 낮아지는 걸 확인했고요.

이번에는 **라쏘 모델까지 함께 놓고** 제대로 비교해 봅시다!

세션에서 배운 망치 비유를 다시 떠올려 볼까요.

> - 🔨 **릿지(L2)** — 기둥에 따라 **다른 힘**으로 누르는 망치. 큰 기둥은 세게, 작은 기둥은 살살.
> - 🔨 **라쏘(L1)** — **모든 기둥을 같은 힘**으로 누르는 망치. 작은 기둥은 못 버티고 사라질 수도.

말로는 알겠는데, 실제로 어떻게 다를까요?

> 💡 **규제 모델은 스케일링이 필수입니다.** 변수마다 단위가 다르면 큰 단위 변수만 억울하게 세게 맞기 때문이에요.
> - **스케일러는 `X_train`으로만 학습**시켜야 합니다.
> - 테스트 데이터의 평균·표준편차까지 알고 있으면 **시험 문제를 미리 본 셈**이 되니까요.

In [ ]:
# 스케일러는 X_train으로만 학습시키고, test에는 transform만 적용합니다.
scaler = StandardScaler().**(X_train_log)
A_train = scaler.transform(X_train_log)
A_test = scaler.**(X_test_log)

rows = []
for a in [0.01, 0.1, 1, 10, 100]:
    ridge = **(alpha=a).fit(A_train, y_train)      # L2 규제 모델
    lasso = **(alpha=a).fit(A_train, y_train)      # L1 규제 모델
    rows.append({
        'alpha': a,
        'Ridge R2': round(r2_score(y_test, ridge.predict(A_test)), 4),
        'Ridge 계수합': round(np.abs(ridge.coef_).sum(), 2),
        'Lasso R2': round(r2_score(y_test, lasso.predict(A_test)), 4),
        'Lasso 계수합': round(np.abs(lasso.coef_).sum(), 2),
        'Lasso 0된 계수': int((np.abs(lasso.coef_) < 1e-8).sum()),
    })

display(pd.DataFrame(rows))

In [ ]:
lasso1 = Lasso(alpha=1).fit(A_train, y_train)

display(pd.DataFrame({
    '변수': X_train_log.columns,
    'Lasso 계수 (alpha=1)': lasso1.coef_.round(3),
}).sort_values('Lasso 계수 (alpha=1)', key=abs, ascending=False))

**Q.** 표를 보고 아래의 질문에 답해 주세요.

1. `alpha`가 커질 때 **릿지**의 계수합과 R²는 어떻게 변하나요?
2. **라쏘**는 `alpha=10`이 어떻게 변하나요? R²가 **음수**가 나온 건 무슨 뜻일까요?
3. 두 모델의 차이를 **망치 비유로** 설명해 봅시다!

**A.**

<details>
<summary><b>▶ 2번의 R² 음수가 이해되지 않는다면</b></summary>
<br>

R²는 **"그냥 평균으로 예측했을 때보다 얼마나 나은가"** 를 재는 지표였습니다.

그럼 R² = 0 이라는 건 **평균으로 예측한 것과 똑같다**는 뜻이겠죠?

모든 계수가 0이 되면 모델은 어떤 값을 내놓게 될까요?

</details>

## B-4. 차원 축소(PCA)

세션에서 다중공선성 대처법으로 **PCA**를 소개했었습니다.

> 💡 상관 있는 변수들을 묶어서 **더 적은 수의 대표 축**으로 바꾸는 것

우리 데이터도 VIF가 5를 넘는 변수가 여럿 있으니 PCA가 도움이 될 수 있을지도 몰라요.

**정말 그럴까요?** 확인해 봅시다.

> ⚠️ **PCA도 스케일링이 필요하고, `X_train`으로만 학습시켜야 합니다.**

In [ ]:
# PCA를 학습시키고 누적 설명분산을 확인해 봅시다. (스케일링된 A_train 사용)
pca_full = PCA().**(A_train)
cum = np.cumsum(pca_full.explained_variance_ratio_)

display(pd.DataFrame({
    '주성분': [f'PC{i+1}' for i in range(len(cum))],
    '설명분산 비율': pca_full.explained_variance_ratio_.round(4),
    '누적 설명분산': cum.round(4),
}))

plt.figure(figsize=(7, 4))
plt.step(range(1, len(cum) + 1), cum, where='mid')
plt.axhline(0.95, color='red', linestyle='--', label='95%')
plt.xlabel('주성분 개수'); plt.ylabel('누적 설명분산'); plt.legend()
plt.show()

**6개 주성분이 분산의 97.4%를 설명**하네요. 95% 기준을 넘으니 6개로 줄여서 모델을 만들어 봅시다.

In [ ]:
# 주성분 6개로 축소한 뒤 선형회귀를 학습시켜 봅시다.
pca6 = PCA(n_components=**).fit(A_train)
P_train = pca6.transform(A_train)
P_test = pca6.**(A_test)          # test는 transform만!

m_pca = LinearRegression().fit(P_train, y_train)
pred_pca = m_pca.predict(P_test)

scores.append(report('B-4. PCA 6성분', pred_pca))
display(pd.DataFrame(scores))

**어라, 성능이 떨어졌네요?** 분산의 97%를 지켰는데 왜 그럴까요?

각 주성분이 **`csMPa`와 얼마나 관련 있는지** 직접 확인해 봅시다.

In [ ]:
Z = pca_full.transform(A_train)
corr_with_y = [abs(np.corrcoef(Z[:, i], y_train)[0, 1]) for i in range(Z.shape[1])]

display(pd.DataFrame({
    '주성분': [f'PC{i+1}' for i in range(len(corr_with_y))],
    '설명분산 비율': pca_full.explained_variance_ratio_.round(3),
    'y와의 상관(절댓값)': np.round(corr_with_y, 3),
}))

**Q.** 위 표를 보고 아래 질문에 답해 주세요.

1. **설명분산이 가장 큰 주성분(PC1)** 은 `csMPa`와 얼마나 관련이 있나요?
2. 반대로 **y와 가장 관련 있는 주성분**은 몇 번인가요? 그 주성분의 설명분산 순위는요?
3. 그렇다면 **PCA가 성능을 떨어뜨린 이유**는 무엇일까요?

**A.**

<details>
<summary><b>▶ 결정적인 힌트</b></summary>
<br>

PCA는 차원을 줄일 때 **무엇을 기준으로** 축을 고를까요?

주성분을 만드는 과정에서 PCA는 **`y`를 참고했을까요?** 세션자료의 PCA 설명을 다시 떠올려 봅시다.


</details>

## B-5.

우리는 총 네 가지 방법으로 모델을 개선해 보려고 했습니다.
결과를 한 자리에 놓고 비교해 봐요.

In [ ]:
final = pd.DataFrame(scores)
display(final)

plt.figure(figsize=(8, 4))
plt.barh(final['모델'], final['test R2'], color='steelblue')
plt.xlabel('test R2'); plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

**Q.** 최종적으로 어떤 모델을 선택할지, **근거와 함께** 적어 주세요.

- 선택한 모델:
- 근거:

그리고 이번 Part B에서 **가장 크게 성능을 올린 시도**는 무엇이었나요?
그 이유와 이번 실습을 통해 얻은 인사이트를 정리해 봅시다.

**A.**

<details>
<summary><b>▶ 무엇을 근거로 삼아야 할까요</b></summary>
<br>

모든 모델은 성능 숫자만이 근거가 되지 않습니다.

**해석 가능성**(회귀계수를 설명할 수 있는가), **단순함**(변수가 적은가), **다중공선성** 같은 것도 함께 봐야 하기 때문이죠.

세션에서 이야기했던 **"분석의 목적이 설명인가 예측인가"** 도 판단 기준이 됩니다.

</details>

---

# Part C · 회귀분석 도전!!


이번에는 새로운 데이터를 가지고, 직접 모델을 설계해 회귀분석을 처음부터 끝까지 수행해 봅시다!

---

## 데이터 소개 — King County 집값

미국 워싱턴주 King County(시애틀 일대)에서 **2014~2015년에 실제로 거래된 주택 21,613건**의 기록이에요.
> **우리는 `price`를 종속변수로 두어 집값을 예측하는 모델을 설계합니다!**

| 칼럼 | 뜻 | | 칼럼 | 뜻 |
| --- | --- | --- | --- | --- |
| `price` | **집값 (종속변수)** | | `sqft_above` | 지하 제외 면적 |
| `date` | 매각 날짜 | | `sqft_basement` | 지하실 면적 |
| `bedrooms` | 침실 수 | | `yr_built` | 건축 연도 |
| `bathrooms` | 욕실 수 | | `yr_renovated` | 리모델링 연도 |
| `sqft_living` | 주거 면적 | | `zipcode` | 우편번호 |
| `sqft_lot` | 부지 면적 | | `lat` / `long` | 위도 / 경도 |
| `floors` | 층수 | | `sqft_living15` | 인근 15채 평균 주거면적 |
| `waterfront` | 물가 조망 여부 (0/1) | | `sqft_lot15` | 인근 15채 평균 부지면적 |
| `view` | 조망 점수 | | `condition` | 상태 등급 |
| `grade` | 주택 등급 | | `id` | 고유 번호 |

---

## 조건

1. **EDA**를 통해 데이터를 파악하고, 눈에 띄는 점을 정리한다
2. 사용할 **독립변수를 직접 고른다** (고른 이유를 밝힐 것)
3. **baseline 모델**을 만든다
4. **모델 개선 방법을 두 가지 이상 시도**하고, 각각 성능이 어떻게 변했는지 기록한다
5. 최종 모델을 정하고, **결론을** 정리한다

> ⚠️ `id`는 고유 번호일 뿐이라 예측에 쓸 수 없으니 반드시 제외해야 합니다!

In [ ]:
df = pd.read_csv('./kc_house_data.csv')   # 경로가 다르면 맞게 수정해 주세요
print(df.shape)
df.head()

## C-1. 데이터 탐색


In [ ]:
# 결측치, 중복, 기술통계 등을 확인해 보세요.




**Q.** 데이터를 살펴보며 눈에 띈 점을 **세 가지 이상** 적어 주세요.

**A.**



## C-2. 변수 선택하기

In [ ]:
# 사용할 독립변수를 골라 보세요.
# 힌트: 상관계수, VIF, 그리고 '이 변수가 의미가 있는가'를 함께 생각해 보세요.




**Q.** 최종적으로 어떤 변수들을 골랐나요? **고른 이유**도 함께 적어 주세요.

**A.**



## C-3. Baseline 모델

In [ ]:
# 데이터를 나누고, baseline 선형회귀 모델을 만들어 보세요.
# 성능은 test 기준으로 R2, MAE, RMSE를 모두 확인하면 좋습니다.




## C-4. 개선 시도 (2가지 이상)

Part B에서 써본 방법들을 떠올려 봅시다. 이 데이터에는 어떤 방법이 적절할까요?

In [ ]:
# 개선 시도 ①




In [ ]:
# 개선 시도 ②




**Q.** 각 시도가 모델의 성능을 어떻게 바꿨나요? **효과가 있었던 것과 없었던 것**을 나눠 정리해 주세요.

| 시도 | test R² | 효과 | 왜 그랬을까 |
| --- | --- | --- | --- |
| baseline | | | |
| 시도 ① | | | |
| 시도 ② | | | |

**A.**



## C-5. 결론

모델을 통해 얻은 결과를 정리해 주세요!

---

**A.**




---

## 마무리

이번 과제에서는 회귀 모델을 직접 만들고, 모델을 개선해 보는 과정을 거쳤습니다.


그리고 모델을 개선할 때 발견했던 것들도 다시 한번 기억해 주세요!

1. **`age` 로그 변환** — 데이터에 맞는 형태를 찾는 것이 복잡한 모델보다 더 적합한 모델일 때가 있습니다. (0.58 → 0.80)
2. **`w/c ratio`** — 도메인 지식으로 만든 변수라도, **이미 있는 정보의 재조합이면** 도움이 되지 않는 경우가 많습니다.
3. **릿지 vs 라쏘** — 데이터에 따라 어떤 규제를 사용할지를 결정해야 해요.
4. **PCA** — 분산을 잘 설명하는 축과 **y를 잘 설명하는 축은 다릅니다.** PCA는 y를 보지 않으니까요!

수고 많으셨습니다! 다음 세션에서 만나요 👋